# 054 — Bandits beyond ε-greedy

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omarahmedZa/ai-ml-course/blob/main/notebooks/054-bandits-beyond-epsilon-greedy.ipynb)

ε-greedy explores at a fixed rate. UCB explores where the uncertainty is. This notebook computes UCB indices by hand, then races the two policies on the same three arms and plots cumulative regret.

## 1. Three UCB indices

The index is `mean + sqrt(2 log t / n)`. At `t = 100`, `2 log t = 9.21`. The least-sampled arm gets the biggest bonus — even when its mean is not the largest.

In [ ]:
import numpy as np                              # array library, preinstalled on Colab

means = np.array([0.50, 0.60, 0.55])            # running averages
counts = np.array([40, 10, 50])                 # pulls so far
t = counts.sum()                                # 100 total pulls
bonus = np.sqrt(2 * np.log(t) / counts)         # shrink with evidence
index = means + bonus                           # UCB score per arm

for i in range(3):                              # print each arm
    print(i + 1, round(float(means[i]), 2), round(float(bonus[i]), 2), round(float(index[i]), 2))
print("pick", int(np.argmax(index)) + 1)        # arm 2, index 1.56

## 2. Regret: UCB versus ε-greedy

Regret is the reward you gave up. Run both policies for 300 pulls on the same arms, compute `t * best_mean - total_reward` at every step, and compare the curves. UCB's should flatten; ε-greedy's should keep climbing.

In [ ]:
import matplotlib.pyplot as plt                 # plotting library, preinstalled on Colab

true_means = np.array([0.45, 0.60, 0.50])       # unknown to the policies
best = true_means.max()                         # 0.60
steps = 300                                     # horizon

def run(policy, eps=0.1, seed=0):               # one simulation
    rng = np.random.default_rng(seed)           # reproducible rewards
    n = np.zeros(3); q = np.zeros(3)            # counts and running means
    total, regret = 0.0, []                     # reward so far, regret trace
    for t in range(1, steps + 1):               # one pull per step
        if policy == "ucb":
            score = q + np.sqrt(2 * np.log(t) / np.maximum(n, 1))
            a = int(np.argmax(score))           # optimistic choice
        else:
            a = int(rng.integers(3)) if rng.random() < eps else int(np.argmax(q))
        r = float(rng.random() < true_means[a])  # 0/1 reward
        n[a] += 1                               # count the pull
        q[a] += (r - q[a]) / n[a]               # update running mean
        total += r                              # accumulate reward
        regret.append(t * best - total)         # what we gave up so far
    return np.array(regret)

plt.figure(figsize=(6, 3))                      # wide, short figure
plt.plot(run("ucb"), label="UCB")                # bonus-driven exploration
plt.plot(run("eps", 0.1), label="eps-greedy 0.1")  # fixed-rate exploration
plt.xlabel("pull")                              # x-axis
plt.ylabel("cumulative regret")                  # y-axis
plt.legend()                                    # name the lines
plt.title("UCB flattens; eps-greedy keeps climbing")   # takeaway
plt.show()                                      # render

## Your turn

Set `eps = 0.5` and rerun section 2: its regret line starts steeper because half of every step is random. Then change `true_means` to `[0.58, 0.60, 0.50]` and rerun. The best arm is now hard to distinguish, so both policies pay more to find it — which one suffers less?